# Invent a Dataset with Adaption

Generate a dataset from scratch for training language models using free-form prompting. No source dataset required to get started.

In this Colab, we will use a detailed `dataset_prompt` to generate a training-ready medical QA datset.

## 1. Install `adaption`

In [ ]:
!pip install adaption

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 159.8/159.8 kB 1.2 MB/s eta 0:00:00


## 2. Instantiate the Adaption Python API client

If you don't have an Adaption API key then refer [this](https://docs.adaptionlabs.ai/introduction/getting-started/#create-an-api-key)  documentation to quickly create one.

In [ ]:
import time
from adaption import Adaption
from google.colab import userdata


client = Adaption(api_key=userdata.get('ADAPTION_API_KEY'))

## 3. Estimate the invention

We will start with a request to estimate the cost of inventing a 100 row dataset. This will allow us to see the credit cost before triggering the full run.



In [ ]:
estimate = client.datasets.invent(
    dataset_prompt=(
        "A dataset of medical question-answer pairs covering diagnoses, treatments, "
        "drug explanations, and healthcare roles."
    ),
    rows=100,
    estimate=True,
)
print(f"Estimated credits: {estimate.estimated_credits} | Available Credits: {estimate.available_credits}",)

Estimated credits: 10.0 | Available Credits: 9013.0


## 4. Invent!

Now that we have priced out the run, we can remove `estimate=True` and create a name for our invented dataset.

Below, we also specify that we would like the output as an `instruction_dataset`. If you would like to invent preference pairs for alignment, pass `training_type="preference_pairs"`.

### Translation & Localization 🌐

Invent also supports expanding the dataset to other languages and locales. Check out the [documentation](https://docs.adaptionlabs.ai/adaptive-data/invent-a-dataset/#expand-across-languages-and-locales) to learn more!

In [ ]:
dataset = client.datasets.invent(
    name="Clinical Q&A",
    dataset_prompt=(
        "A dataset of medical question-answer pairs covering diagnoses, treatments, "
        "drug explanations, and healthcare roles."
    ),
    rows=100,
    training_type="instruction_dataset",
)
dataset_id = dataset.id
print(dataset_id, dataset.status)  # running

86f2ba53-3ab3-4abb-b917-d6364ead105c running


## 5. Wait for invention to complete

We run the invention process asynchonously on our platform. Below we use the `wait_for_completion` helper to poll the status of the dataset until it's ready to be used.

In [ ]:
record = client.datasets.wait_for_completion(dataset_id)
print(f"Generated {record.row_count} rows")


Generated 100 rows


## 6. Download the dataset

After the dataset is ready, we can download it to a local `jsonl` file for inspection and usage in our own products later on.

In [ ]:
client.datasets.download(
    dataset_id,
    file_format="jsonl",
).write_to_file("clinical-qa.jsonl")

## 7. View the datatset

In [ ]:
import pandas as pd
from google.colab import files
from google.colab.data_table import DataTable

df = pd.read_json('clinical-qa.jsonl', lines=True)
DataTable(df, include_index=False)

,enhanced_prompt,enhanced_completion
0,In the context of post-autologous hematopoieti...,Following autologous hematopoietic stem cell t...
1,A 45-year-old male presents with a history of ...,Based on the clinical presentation of chronic ...
2,Analyze the differential diagnosis for a patie...,***Disclaimer:** I am an artificial intelligen...
3,Focusing on spinal trauma recovery and pressur...,"In the context of spinal trauma recovery, pres..."
4,Explain the difference between a Pharmacist an...,### The Short Answer: Terminology vs. Role\n\n...
...,...,...
95,Integrating rigorous diagnostic frameworks wit...,### Clinical Case Analysis: Differentiating De...
96,Compare and contrast the roles and scopes of p...,"In the United States primary care setting, Nur..."
97,"Describe the pharmacodynamics of Digoxin, incl...",### Pharmacodynamics of Digoxin\n\nDigoxin is ...
98,Drawing on the pathophysiological differences ...,### Pathophysiological Foundations and Diagnos...


## 8. Evaluate dataset diversity with DCScore

The [DCScore](https://arxiv.org/abs/2502.08512) is a dataset diversity metric that treats diversity evaluation as a sample classification task over embeddings. The score ranges from `1` (every row identical) to `N` (every row fully distinct), so we also report it normalized by `N`.

We fetch the reference implementation from GitHub and embed rows with `all-mpnet-base-v2`, the default text embedder in the DCScore paper.

In [ ]:
!git clone -q https://github.com/BlueWhaleLab/DCScore
!pip install -q sentence-transformers

In [ ]:
import sys
sys.path.append("DCScore")
from dcscore_function import DCScore


def evaluate_diversity(texts, model="sentence-transformers/all-mpnet-base-v2", batch_size=128, tau=0.1):
    """Return DCScore for a list of texts. 1 = all identical, len(texts) = all distinct."""
    evaluator = DCScore(model)
    embeddings, n, _ = evaluator.get_embedding(texts, batch_size=batch_size)
    score = evaluator.calculate_dcscore_by_embedding(embeddings, kernel_type="cs", tau=tau)
    return {"dcscore": score, "normalized": score / n, "n": n}

In [ ]:
import json
rows = []
with open("clinical-qa.jsonl", "r", encoding="utf-8") as file:
    for line in file:
        if line.strip():  # Skip empty lines
            rows.append(json.loads(line))

data = [f"{r['enhanced_prompt']}\n\n{r['enhanced_completion']}" for r in rows]
result = evaluate_diversity(data)
print(f"Diveristy score: {result['dcscore']:.1f} / {result['n']} rows  (normalized {result['normalized']:.3f})")

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Diveristy score: 89.9 / 100 rows  (normalized 0.899)
